# Bronze FINESS Data Exploration

This notebook explores the **raw FINESS data** from the Bronze layer.

FINESS = Fichier National des Établissements Sanitaires et Sociaux (French healthcare facility registry)

In [1]:
import pandas as pd
import numpy as np
import sys
sys.path.append('..')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

## 1. Load Raw FINESS Data

The raw file has:
- A metadata header row (skipped)
- A 'structureet' column as first column (removed)
- 31 data columns total

In [2]:
# Load raw data exactly as the pipeline does
df_bronze = pd.read_csv(
    '../data/bronze/finess/finess.csv',
    sep=';',
    skiprows=1,  # Skip metadata header
    header=None,
    dtype=str,
    encoding='utf-8'
)

print(f"📊 Shape before cleanup: {df_bronze.shape}")
print(f"First column (structureet): {df_bronze.iloc[0, 0]}")

📊 Shape before cleanup: (204916, 32)
First column (structureet): structureet


In [3]:
# Remove structureet column
df_bronze = df_bronze.iloc[:, 1:]

# Apply column names (official schema from data.gouv.fr)
# Source: https://www.data.gouv.fr/fr/datasets/finess-extraction-du-fichier-des-etablissements/
FINESS_COLUMNS = [
    "nofinesset",           # Numéro FINESS ET
    "nofinessej",           # Numéro FINESS EJ
    "rs",                   # Raison sociale
    "rslongue",             # Raison sociale longue
    "complrs",              # Complément de raison sociale
    "compldistrib",         # Complément de distribution
    "numvoie",              # Numéro de voie
    "typvoie",              # Type de voie
    "voie",                 # Libellé de voie
    "compvoie",             # Complément de voie
    "lieuditbp",            # Lieu-dit / BP
    "commune",              # Code Commune
    "departement",          # Département
    "libdepartement",       # Libellé département
    "ligneacheminement",    # Code Postal + Lib commune
    "telephone",            # Téléphone
    "telecopie",            # Télécopie
    "categetab",            # Catégorie d'établissement
    "libcategetab",         # Libellé catégorie d'établissement
    "categagretab",         # Catégorie d'agrégat d'établissement
    "libcategagretab",      # Libellé catégorie d'agrégat d'établissement
    "siret",                # Numéro de SIRET (position 23 in docs)
    "codeape",              # Code APE
    "codemft",              # Code MFT
    "libmft",               # Libellé MFT
    "codesph",              # Code SPH
    "libsph",               # Libellé SPH
    "dateouv",              # Date d'ouverture
    "dateautor",            # Date d'autorisation
    "datemaj",              # Date de mise à jour
    "numuai"                # Numéro éducation nationale
]

df_bronze.columns = FINESS_COLUMNS[:len(df_bronze.columns)]

print(f"\n📊 Final Shape: {df_bronze.shape}")
print(f"\n📋 Columns ({len(df_bronze.columns)}):")
for i, col in enumerate(df_bronze.columns, 1):
    print(f"  {i:2}. {col}")


📊 Final Shape: (204916, 31)

📋 Columns (31):
   1. nofinesset
   2. nofinessej
   3. rs
   4. rslongue
   5. complrs
   6. compldistrib
   7. numvoie
   8. typvoie
   9. voie
  10. compvoie
  11. lieuditbp
  12. commune
  13. departement
  14. libdepartement
  15. ligneacheminement
  16. telephone
  17. telecopie
  18. categetab
  19. libcategetab
  20. categagretab
  21. libcategagretab
  22. siret
  23. codeape
  24. codemft
  25. libmft
  26. codesph
  27. libsph
  28. dateouv
  29. dateautor
  30. datemaj
  31. numuai


## 2. Data Overview

In [4]:
# Basic info
print("📈 Dataset Statistics:")
print(f"  Total rows: {len(df_bronze):,}")
print(f"  Total columns: {len(df_bronze.columns)}")
print(f"  Memory usage: {df_bronze.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# Display first few rows
print("\n📋 First 5 Rows:")
df_bronze.head()

📈 Dataset Statistics:
  Total rows: 204,916
  Total columns: 31
  Memory usage: 283.91 MB

📋 First 5 Rows:


,nofinesset,nofinessej,rs,rslongue,complrs,compldistrib,numvoie,typvoie,voie,compvoie,lieuditbp,commune,departement,libdepartement,ligneacheminement,telephone,telecopie,categetab,libcategetab,categagretab,libcategagretab,siret,codeape,codemft,libmft,codesph,libsph,dateouv,dateautor,datemaj,numuai
0,010000024,010780054,CH DE FLEYRIAT,CENTRE HOSPITALIER DE BOURG-EN-BRESSE FLEYRIAT,NaN,NaN,900,RTE,DE PARIS,NaN,NaN,451,01,AIN,01440 VIRIAT,0474454647,0474454114,355,Centre Hospitalier (C.H.),1102,Centres Hospitaliers,26010004500012,8610Z,03,ARS établissements Publics de santé dotation g...,1,Etablissement public de santé,1979-02-13,1979-02-13,2020-02-04,NaN
1,010000032,010780062,CH BUGEY SUD,CENTRE HOSPITALIER BUGEY SUD,NaN,NaN,700,AV,DE NARVIK,NaN,BP 139,034,01,AIN,01300 BELLEY,0479425959,0479425996,355,Centre Hospitalier (C.H.),1102,Centres Hospitaliers,26010003700068,8610Z,03,ARS établissements Publics de santé dotation g...,1,Etablissement public de santé,1901-01-01,1901-01-01,2021-07-07,NaN
2,010000065,010780096,CH DE TREVOUX - MONTPENSIER,CENTRE HOSPITALIER DE TREVOUX - MONTPENSIER,NaN,NaN,14,R,DE L'HOPITAL,NaN,BP 615,427,01,AIN,01606 TREVOUX CEDEX,0474105000,0474105019,355,Centre Hospitalier (C.H.),1102,Centres Hospitaliers,26010028400017,8610Z,03,ARS établissements Publics de santé dotation g...,1,Etablissement public de santé,1901-01-01,1901-01-01,2018-01-12,NaN
3,010000081,010780112,CH DU PAYS DE GEX,CENTRE HOSPITALIER DU PAYS DE GEX,NaN,NaN,160,R,MARC PANISSOD,NaN,BP 437,173,01,AIN,01174 GEX CEDEX,0450403838,0450403825,106,"Centre hospitalier, ex Hôpital local",1106,Hôpitaux Locaux,26010010200011,8610Z,03,ARS établissements Publics de santé dotation g...,1,Etablissement public de santé,1901-01-01,1901-01-01,2020-02-04,NaN
4,010000099,010780120,CH DE MEXIMIEUX,CENTRE HOSPITALIER DE MEXIMIEUX,NaN,NaN,13,AV,DU DOCTEUR BOYER,NaN,NaN,244,01,AIN,01800 MEXIMIEUX,0474610177,0474347184,106,"Centre hospitalier, ex Hôpital local",1106,Hôpitaux Locaux,26010013600019,8610Z,03,ARS établissements Publics de santé dotation g...,1,Etablissement public de santé,1945-01-01,1945-01-01,2020-06-30,NaN


## 3. Column Analysis

In [5]:
# Null value analysis
null_counts = df_bronze.isnull().sum()
null_pct = (null_counts / len(df_bronze) * 100).round(2)

null_summary = pd.DataFrame({
    'Column': df_bronze.columns,
    'Null_Count': null_counts.values,
    'Null_Percentage': null_pct.values
}).sort_values('Null_Percentage', ascending=False)

print("🔍 Null Value Analysis:")
print(null_summary[null_summary['Null_Percentage'] > 0])

🔍 Null Value Analysis:
               Column  Null_Count  Null_Percentage
30             numuai      201547            98.36
9            compvoie      200947            98.06
10          lieuditbp      193984            94.67
26             libsph      193062            94.22
25            codesph      193062            94.22
5        compldistrib      193031            94.20
22            codeape      158961            77.57
16          telecopie      157496            76.86
6             numvoie      118210            57.69
15          telephone      117157            57.17
21              siret      114386            55.82
7             typvoie      107403            52.41
8                voie      104433            50.96
28          dateautor      103093            50.31
11            commune      102458            50.00
19       categagretab      102458            50.00
20    libcategagretab      102458            50.00
17          categetab      102458            50.00
18      

## 4. Sample Records

In [6]:
# Sample random establishments
print("🏥 Sample Establishments (Random 3):")
df_bronze.sample(3).T

🏥 Sample Establishments (Random 3):


,7446,82957,151051
nofinesset,120002910,800019135,540026960
nofinessej,120002902,800017659,945475.4
rs,PHARMACIE CROZATIER-CLANET,SAAAS-SAFEP APAJH80 PONT-DE-METZ,6837942.9
rslongue,PHARMACIE CROZATIER-CLANET,NaN,"1,ATLASANTE,95,BAN,EPSG:2154 RGF93 / Lambert-9..."
complrs,NaN,NaN,2026-01-05
compldistrib,NaN,NaN,NaN
numvoie,3,2,NaN
typvoie,AV,ALL,NaN
voie,DU CARDINAL VERDIER,MARC SIBERCHICOT,NaN
compvoie,NaN,NaN,NaN


## 5. Key Field Distributions

In [7]:
# Category distributions
print("📊 Top 10 Establishment Categories:")
if 'libmft' in df_bronze.columns:
    print(df_bronze['libmft'].value_counts().head(10))

print("\n📊 Top 10 Regions:")
if 'libregion' in df_bronze.columns:
    print(df_bronze['libregion'].value_counts().head(10))

📊 Top 10 Establishment Categories:
libmft
Etablissement Tarif Libre                                       42182
Président du Conseil Départemental                              12915
Indéterminé                                                      8108
ARS établissements Publics de santé dotation globale             5499
ARS / DG dotation globale                                        3406
Tarifs conventionnels assurance maladie                          3365
ARS /ARS PCD   Dotation forfait ou prix de journée globalisé     3271
ARS/PCD, Tarif partiel, habilité aide sociale sans PUI           3148
ARS établissements médico-soc. non financés dotation globale     2374
ARS établissements de santé non financés dotation globale        2046
Name: count, dtype: int64

📊 Top 10 Regions:


In [8]:
# Data types summary
print("\n📝 Data Types:")
print(df_bronze.dtypes.value_counts())


📝 Data Types:
str    31
Name: count, dtype: int64
